# Metrics to implement
## metric list
- AUC
- GAUC
- HitRates
- Recall
- Precision
- F1-scores
- NDCG

## End-to-End Metrics

The metrics measure the all pipeline(regard  the pipeline from recall to ranking as a single module)

references:  https://www.kimi.com/share/19df7155-3bf2-837b-8000-0000a423895b


### AUC
meaning:  the possibilty of a sampled positive sample score > a sampled negative sample score

In [ ]:
def cal_auc(pred, label):
    """
    pred: [B, ], 1D-list 
    label: [B, ], 1D- list 
    return: int
    method: possibility
    """

    pos_list = []
    neg_list = []
    for i in range(len(label)):
        if label[i] == 1:
            pos_list.append(i)
        else:
            neg_list.append(i)

    pos_cnt, neg_cnt = len(pos_list), len(neg_list)
    pos_over_neg_cnt = 0
    for p_idx in pos_list:
        for n_idx in neg_list:
            if  pred[p_idx] > pred[n_idx]:
                pos_over_neg_cnt += 1
            elif pred[p_idx] == pred[n_idx]:
                pos_over_neg_cnt += 0.5
    res = 0.5 if pos_cnt==0 or neg_cnt==0 else pos_over_neg_cnt/(pos_cnt * neg_cnt)
    return res



pred= [0.1, 0.2, 0.3, 0.4, 0.5]
label = [1, 0, 0, 1, 1 ]
#auc = 4/6 = 0.66
cal_auc(pred, label)



0.6666666666666666

## GAUC
meaning: separate users into different groups. Calculate auc of each group and then use weighted sum over all auc values

In [ ]:
def cal_gauc(pred, label):
    """
    w =  number of impression
    pred: [G, L] G= number of group , L = group size
    label: [G, L] G= number of group , L = group size
    """

    tot_cnt = 0
    auc_sum = 0
    for i in range(len(label)):
        auc_res = cal_auc(pred[i], label[i])    
        print(auc_res, len(label[i]))
        auc_res *= len(label[i])
        tot_cnt += len(label[i])
        auc_sum += auc_res
    res = auc_sum/tot_cnt if tot_cnt > 0 else 0.5
    return res


pred2= [[0.1, 0.2, 0.3, 0.4, 0.5], [0.6, 0.7, 0.8, 0.9]]
label2 = [[1, 0, 0, 1, 1 ], [0, 1, 0, 1 ]]
#auc = 4/6 = 0.66
#auc2 = 3/4 = 0.75
# gauc = 0.75 * 5/9 + 0.75 * 4/9  
cal_gauc(pred2, label2), 0.66 * 5/9 + 0.75 * 4/9


0.6666666666666666 5
0.75 4


(0.7037037037037037, 0.7)

## HitRate@N
meaning: measure if topN results is valid or not. 验证返回/检索结果 **是否有用， 0或1**

formula:  HitRate@3 = (top3中至少有一个点击的请求数) / 100

example： 
- 100 items returned to 1 user request.  If user click one of topN items -> Hitrate=1 otherwise, hitrate=0
- For 100 user requests, each obtains 100 items. only 3 lists obtained click on top3 items.   Hitrate = 3/100

In [ ]:

def cal_hitrate(pred, label, topK=3):
    """
    pred: [B, L], each sample = one request 
    label: [B, L], 
    return: int
    method: request's top3 results are hit or not hit
    """
    hit = 0
    total = 0
    for i in range(len(label)):
        sc = [ (pred[i][j], j) for j in range(len(pred[i]))]
        sc = sorted(sc, key=lambda x: x[0], reverse=True)
        hit +=  1 if sum(1 for v in sc[:topK] if label[i][v[1]] == 1) >0 else 0
        total += 1
    return hit / total if total > 0 else 0


pred2= [[0.1, 0.2, 0.3, 0.4, 0.5], [0.6, 0.7, 0.8, 0.9]]
label2 = [[1, 0, 0, 1, 1 ], [1, 0, 0, 0 ]]
#hitrate@3: (1+0)/2 = 0.5 
cal_hitrate(pred2, label2), 0.5



(0.5, 0.5)

## Recall
meaning: coverage of expected positive results (目标正样本的覆盖率， 或者对正样本找齐没有)

formula: Recall@3 = (所有请求top3中被点击的总数) / (用户实际点击的总数)

example:
- 你需要知道用户总共点击了多少个item（不只是top3里的）

| 请求  | top3中点击数 | 该请求用户总点击数 |
| --- | -------- | --------- |
| 1   | 1        | 2         |
| 2   | 0        | 1         |
| 3   | 2        | 3         |
| ... | ...      | ...       |
| 100 | 1        | 2         |

- top3命中总数 = 80
- 至少命中1个的请求数 = 60
- 用户总点击数 = 150
- Recall@3 = 80/150 = 53.3%


In [ ]:

def cal_recall(pred, label, topK=3):
    """
    pred: [B, L], each sample = one request 
    label: [B, L], 
    return: int
    method: possibility
    """
    tot_click = 0
    total = 0
    for i in range(len(label)):
        sc = [ (pred[i][j], j) for j in range(len(pred[i]))]
        sc = sorted(sc, key=lambda x: x[0], reverse=True)
        tot_click += sum([label[i][v[1]] for v in sc[:topK]])
        print(tot_click)
        total += sum(label[i])
        print(tot_click, "total: ",total)
    return tot_click / total if total > 0 else 0


pred2= [[0.1, 0.2, 0.3, 0.4, 0.5], [0.6, 0.7, 0.8, 0.9]]
label2 = [[1, 0, 0, 1, 1 ], [1, 0, 0, 1 ]]
#recall@3: (3)/5 = 0.6
cal_recall(pred2, label2), 3/5



2
2 total:  3
3
3 total:  5


(0.6, 0.6)

## Precision

meaning: purity of result list. or how precision the result is

formula:
- Precision@3 = (所有请求top3中被点击的总数) / (100 × 3)
- 需要展开所有request结果list 再合并统计
- **这个K 和 N 在不同场景定义不一样**
- 仅排序场景： N=排序返回结果数， 比如N=100， k=topK
- 但是如果是**端到端 (从召回到排序结果)， N=召回打分数(即输入)， k=排序结果里面要的topK数**

Example:
- 例子：
    - 请求1的top3：被点击了1个
    - 请求2的top3：被点击了2个
    - ...
    - 100个请求累计被点击了80个
    - Precision@3 = 80 / 300 = 26.7%




In [24]:

def cal_precision(pred, label, topK=3):
    """
    pred: [B, L], each sample = one request 
    label: [B, L], 
    return: int
    method: possibility
    """
    tot_click = 0
    total = 0
    for i in range(len(label)):
        sc = [ (pred[i][j], j) for j in range(len(pred[i]))]
        sc = sorted(sc, key=lambda x: x[0], reverse=True)
        tot_click += sum([label[i][v[1]] for v in sc[:topK]])
        print(tot_click)
        total += len(pred[i])
        print(tot_click, "total: ",total)
    return tot_click / total if total > 0 else 0


pred2= [[0.1, 0.2, 0.3, 0.4, 0.5], [0.6, 0.7, 0.8, 0.9]]
label2 = [[1, 0, 0, 1, 1 ], [1, 0, 0, 1 ]]
#recall@3: (3)/(5+4) = 1/3
cal_precision(pred2, label2), 1/3



2
2 total:  5
3
3 total:  9


(0.3333333333333333, 0.3333333333333333)

## NDCG/逆序对

meaning: similarity between ranking order by rank score and ideal ranking order by relevence label

formula: DCG/IDCG
- DCG = sum( topk_label_sorted_by_rankscore/log2(position+1) )
- IDCG = DCG computed by relevenceScore

对于每个请求：
1. 计算 DCG@3 = Σ(rel_i / log₂(i+1))
2. 计算 IDCG@3 = 把点击的item放最前面，再算DCG
3. 计算 NDCG@3 = DCG / IDCG

最终 NDCG@3 = 所有请求NDCG@3的平均值


In [28]:

def cal_ndcg(pred, label, topK=3):
    """
    pred: [B, L], each sample = one request 
    label: [B, L], 
    return: int
    method: possibility
    """
    def cal_dcg(pred, label, topK=3):
        import math
        dcg = 0
        for i in range(len(label)):
            sc = [ (pred[i][j], j) for j in range(len(pred[i]))]
            sc = sorted(sc, key=lambda x: x[0], reverse=True)
            dcg += sum([label[i][v[1]]/math.log2(pos+1 +1) for pos, v in enumerate(sc[:topK])] )
        return dcg

    dcg_res = cal_dcg(pred, label, topK)    
    idcg_res = cal_dcg(label, label, topK)
    # normalized by ideal ndcg to reduce effect of different lengths of list
    res = dcg_res/idcg_res if idcg_res > 0 else 0.5
    return res

pred2= [[0.1, 0.2, 0.3, 0.4, 0.5], [0.6, 0.7, 0.8, 0.9]]
label2 = [[1, 0, 0, 1, 1 ], [1, 0, 0, 1 ]]
#recall@3: (3)/(5+4) = 1/3
cal_ndcg(pred2, label2, 3), 



(0.6993694869720468,)